# Raman Spectroscopy — Consensus Spectra

Dataset: `combined_raman_spectra.xlsx`

**Pipeline:**
1. Load & parse data
2. Remove spectra with non-standard laser wavelengths (633 nm & 785 nm)
3. Preprocessing: Savitzky-Golay smoothing → ALS baseline correction → L² normalisation
4. Save preprocessed dataset
5. Convert to functional data (B-spline basis expansion)
6. Consensus spectra: Arithmetic Mean · B-spline Mean · Robust Median Centre · Wasserstein Barycenter

> **Data availability:** The raw LabRAM `.txt` spectra are not included in this repository because the final datasets are provided in `data/`. This notebook reads `combined_raman_spectra.xlsx`, compiled from those spectra; the resulting consensus spectra are provided (aligned) as `data/raman_consensus_spectra_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.signal import savgol_filter
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from scipy.interpolate import BSpline, make_lsq_spline
from scipy.stats import median_abs_deviation

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# Optional: POT for Wasserstein barycenter
try:
    import ot
    HAS_POT = True
    print('POT (Python Optimal Transport) available.')
except ImportError:
    HAS_POT = False
    print('POT not found — Wasserstein barycenter will use iterative Sinkhorn approximation via numpy.')

## 1. Load & Remove Non-Standard Laser Spectra

In [ ]:
df = pd.read_excel('combined_raman_spectra.xlsx')
raman_shift = df['Raman Shift'].values

print(f'Total columns (before filtering): {df.shape[1] - 1} spectra')
print(f'Raman shift range: {raman_shift.min():.1f} – {raman_shift.max():.1f} cm⁻¹')
print(f'Data points: {len(raman_shift)}')

# Identify and drop spectra acquired at 633 nm or 785 nm
exclude = [c for c in df.columns if '633nm' in c or '785nm' in c]
print(f'\nDropping {len(exclude)} non-standard laser spectra:')
for e in exclude:
    print(f'  • {e}')

df_filtered = df.drop(columns=exclude)
spectra_raw = df_filtered.drop(columns=['Raman Shift'])

print(f'\nRemaining spectra: {spectra_raw.shape[1]}')
print(f'Missing values: {spectra_raw.isna().sum().sum()}')

## 2. Preprocessing Pipeline

Identical to `raman_eda.ipynb`:
1. **Savitzky-Golay smoothing** (window=5, polyorder=2)
2. **ALS baseline correction** (λ=1×10⁵, p=0.01, 10 iterations)
3. **L² normalisation**

In [ ]:
def sg_smooth(y, window=5, polyorder=2):
    return savgol_filter(y, window_length=window, polyorder=polyorder)

def als_baseline(y, lam=1e5, p=0.01, n_iter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, format='csc')
        Z = spsolve(W + H, w * y)
        w = np.where(y > Z, p, 1 - p)
    return Z

def l2_normalize(y):
    norm = np.linalg.norm(y)
    return y / norm if norm > 0 else y

preprocessed = {}
for col in spectra_raw.columns:
    y = spectra_raw[col].values.astype(float)
    y = sg_smooth(y)
    y = y - als_baseline(y)
    y = np.clip(y, 0, None)
    y = l2_normalize(y)
    preprocessed[col] = y

spectra = pd.DataFrame(preprocessed, index=raman_shift)
spectra.index.name = 'Raman Shift'

print('Preprocessing complete.')
print(f'Shape: {spectra.shape}  |  Any NaN: {spectra.isna().any().any()}')

## 3. Save Preprocessed Dataset

In [ ]:
out_path = 'raman_preprocessed.xlsx'
spectra.to_excel(out_path)
print(f'Saved preprocessed dataset → {out_path}')
print(f'Shape: {spectra.shape[0]} wavenumbers × {spectra.shape[1]} spectra')

## 4. Overview Plot — All Preprocessed Spectra

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = cm.tab10(np.linspace(0, 1, spectra.shape[1]))
for i, col in enumerate(spectra.columns):
    ax.plot(raman_shift, spectra[col], alpha=0.45, linewidth=0.7, color=colors[i])
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title(f'All {spectra.shape[1]} Preprocessed Spectra (532 nm only)')
plt.tight_layout()
plt.show()

## 5. Convert to Functional Data (B-spline Basis Expansion)

Each discrete spectrum is represented as a continuous function by fitting a B-spline basis expansion. This converts the data from a vector of 2118 intensity values to a smooth functional object defined over the Raman shift domain.

**Settings:** cubic B-splines (order 4), interior knots placed at every 10th data point.

In [ ]:
x = raman_shift.astype(float)
n_pts = len(x)

# Interior knots: every 10th point (≈212 knots), excluding boundary
interior_knots = x[10:-10:10]
k = 3  # cubic spline order
t = np.concatenate(([x[0]] * (k + 1), interior_knots, [x[-1]] * (k + 1)))

functional_coeffs = {}   # stores spline coefficients per spectrum
functional_curves = {}   # stores evaluated smooth curve per spectrum

for col in spectra.columns:
    y = spectra[col].values
    spl = make_lsq_spline(x, y, t, k=k)
    functional_coeffs[col] = spl.c
    functional_curves[col] = spl(x)

functional_df = pd.DataFrame(functional_curves, index=raman_shift)
functional_df.index.name = 'Raman Shift'

print(f'B-spline basis: {len(t) - k - 1} basis functions, order {k+1}')
print(f'Functional data shape: {functional_df.shape}')

In [ ]:
# Visual check: raw preprocessed vs functional representation for 3 example spectra
examples = list(spectra.columns[:3])
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, examples):
    ax.plot(x, spectra[col].values, alpha=0.5, linewidth=0.8, color='steelblue', label='Preprocessed')
    ax.plot(x, functional_df[col].values, linewidth=1.2, color='crimson', label='B-spline fit')
    ax.set_title(col.split()[0] + '\n' + col.split()[1] if len(col.split()) > 1 else col, fontsize=8)
    ax.set_xlabel('Raman Shift (cm⁻¹)')
axes[0].set_ylabel('Normalised Intensity (a.u.)')
axes[0].legend(fontsize=8)
plt.suptitle('Functional Representation: B-spline Fit vs Preprocessed Data', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Consensus Spectra

Four consensus methods are computed on the **functional (B-spline smoothed)** representations:

| Method | Description |
|--------|-------------|
| **Arithmetic Mean** | Pointwise average — simple, sensitive to outliers |
| **B-spline Mean** | Mean of B-spline coefficient vectors, re-evaluated on the grid |
| **Robust Median Centre** | Pointwise median with MAD-based uncertainty — outlier-resistant |
| **Wasserstein Barycenter** | Barycenter in the space of probability measures (optimal transport) |

### 6.1 Arithmetic Mean

In [ ]:
F = functional_df.values  # shape: (n_pts, n_spectra)

arith_mean = F.mean(axis=1)
arith_std  = F.std(axis=1)

fig, ax = plt.subplots(figsize=(14, 5))
for col in functional_df.columns:
    ax.plot(x, functional_df[col], alpha=0.25, linewidth=0.6, color='steelblue')
ax.plot(x, arith_mean, color='crimson', linewidth=2, label='Arithmetic Mean')
ax.fill_between(x, arith_mean - arith_std, arith_mean + arith_std,
                color='crimson', alpha=0.15, label='±1 SD')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Consensus Spectrum — Arithmetic Mean ± 1 SD')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Arithmetic mean — peak at {x[np.argmax(arith_mean)]:.1f} cm⁻¹')

### 6.2 B-spline Mean (Mean in Coefficient Space)

In [ ]:
# Average the spline coefficients, then evaluate the mean spline on the grid
coeff_matrix = np.stack(list(functional_coeffs.values()), axis=1)  # (n_basis, n_spectra)
mean_coeffs  = coeff_matrix.mean(axis=1)

mean_spline = BSpline(t, mean_coeffs, k)
bspline_mean = mean_spline(x)
bspline_std  = np.std([BSpline(t, coeff_matrix[:, i], k)(x) for i in range(coeff_matrix.shape[1])], axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
for col in functional_df.columns:
    ax.plot(x, functional_df[col], alpha=0.25, linewidth=0.6, color='steelblue')
ax.plot(x, bspline_mean, color='darkorange', linewidth=2, label='B-spline Mean')
ax.fill_between(x, bspline_mean - bspline_std, bspline_mean + bspline_std,
                color='darkorange', alpha=0.15, label='±1 SD')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Consensus Spectrum — B-spline Mean (Coefficient Space Average) ± 1 SD')
ax.legend()
plt.tight_layout()
plt.show()

print(f'B-spline mean — peak at {x[np.argmax(bspline_mean)]:.1f} cm⁻¹')

### 6.3 Robust Median Centre

In [ ]:
robust_median = np.median(F, axis=1)
robust_mad    = median_abs_deviation(F, axis=1)

def geometric_median(X, tol=1e-6, max_iter=300):
    """Weiszfeld algorithm for the geometric (L1) median of columns of X."""
    # X: (n_pts, n_spectra)
    y = X.mean(axis=1).copy()
    for _ in range(max_iter):
        dists = np.linalg.norm(X - y[:, None], axis=0)  # (n_spectra,)
        dists = np.where(dists < 1e-10, 1e-10, dists)
        weights = 1.0 / dists
        y_new = (X * weights[None, :]).sum(axis=1) / weights.sum()
        if np.linalg.norm(y_new - y) < tol:
            break
        y = y_new
    return y_new

geo_median = geometric_median(F)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)


ax = axes[0]
for col in functional_df.columns:
    ax.plot(x, functional_df[col], alpha=0.2, linewidth=0.6, color='steelblue')
ax.plot(x, robust_median, color='green', linewidth=2, label='Pointwise Median')
ax.fill_between(x, robust_median - robust_mad, robust_median + robust_mad,
                color='green', alpha=0.18, label='±1 MAD')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Robust Median Centre — Pointwise Median ± 1 MAD')
ax.legend()

# Panel 2: geometric (L1) median
ax = axes[1]
for col in functional_df.columns:
    ax.plot(x, functional_df[col], alpha=0.2, linewidth=0.6, color='steelblue')
ax.plot(x, geo_median, color='darkgreen', linewidth=2, label='Geometric (L1) Median')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_title('Robust Median Centre — Geometric (L1) Median')
ax.legend()

plt.suptitle('Consensus Spectrum — Robust Median Centre', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Pointwise median — peak at {x[np.argmax(robust_median)]:.1f} cm⁻¹')
print(f'Geometric median — peak at {x[np.argmax(geo_median)]:.1f} cm⁻¹')

### 6.4 Wasserstein Barycenter (Sinkhorn, reg = 1×10⁻⁴)

Each spectrum is treated as a discrete probability measure over the Raman shift axis (shifted non-negative and sum-normalised). The regularised Wasserstein barycenter is computed via the **Sinkhorn algorithm** (Cuturi & Doucet 2014) using POT, with entropic regularisation reg = 1×10⁻⁴.

The cost matrix is the squared Euclidean distance on the (min-max normalised) wavenumber axis. The grid is subsampled by a factor of 5 (≈424 points) for tractability and the result is interpolated back to the original 2118-point grid.

In [ ]:
import ot

# ── Subsample grid for tractability ──────────────────────────────────────────
step = 5
x_sub  = x[::step]                     # ~424 points
F_sub  = F[::step, :]                  # (n_sub, n_spectra)
n_sub  = len(x_sub)

# ── Cost matrix on normalised wavenumber axis ─────────────────────────────────
x_norm = (x_sub - x_sub.min()) / (x_sub.max() - x_sub.min())
M = ot.dist(x_norm.reshape(-1, 1), x_norm.reshape(-1, 1), metric='sqeuclidean')
# M lies in [0, 1]; with reg=1e-4 the kernel exp(-M/reg) underflows for standard
# Sinkhorn, so we use the log-domain stabilised variant.

# ── Convert each spectrum to a valid probability histogram ────────────────────
def to_hist(y):
    y = np.clip(y, 0, None)
    y = y - y.min()
    s = y.sum()
    return y / s if s > 0 else np.ones(len(y)) / len(y)

A = np.stack([to_hist(F_sub[:, j]) for j in range(F_sub.shape[1])], axis=1)  # (n_sub, n_spectra)

# ── Sinkhorn barycenter — log-domain stabilised, reg = 1e-4 ──────────────────
weights = np.ones(A.shape[1]) / A.shape[1]

print(f'Running stabilised Sinkhorn barycenter on {n_sub} grid points × {A.shape[1]} spectra (reg=1e-4) ...')
bary_hist = ot.bregman.barycenter_stabilized(
    A, M, reg=1e-4,
    weights=weights,
    numItermax=2000,
    stopThr=1e-9,
)

# ── Interpolate back to full 2118-point grid and L²-normalise ─────────────────
wasserstein_bary = np.interp(x, x_sub, bary_hist)
wasserstein_bary = np.clip(wasserstein_bary, 0, None)
wasserstein_bary = l2_normalize(wasserstein_bary)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
for col in functional_df.columns:
    ax.plot(x, functional_df[col], alpha=0.2, linewidth=0.6, color='steelblue')
ax.plot(x, wasserstein_bary, color='purple', linewidth=2,
        label='Wasserstein Barycenter (Sinkhorn stabilised, reg=1e-4)')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Consensus Spectrum — Wasserstein Barycenter (Sinkhorn, reg = 1×10⁻⁴)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Wasserstein barycenter — peak at {x[np.argmax(wasserstein_bary)]:.1f} cm⁻¹')

### 6.5 Comparison: All Four Consensus Methods

In [ ]:
consensus = {
    'Arithmetic Mean':        arith_mean,
    'B-spline Mean':          bspline_mean,
    'Robust Median Centre':   robust_median,
    'Geometric Median':       geo_median,
    'Wasserstein Barycenter': wasserstein_bary,
}
colors_c = ['crimson', 'darkorange', 'green', 'darkgreen', 'purple']

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Top: all five overlaid
ax = axes[0]
for (label, curve), color in zip(consensus.items(), colors_c):
    ax.plot(x, curve, linewidth=1.8, color=color, label=label)
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('All Five Consensus Spectra — Overlay')
ax.legend(fontsize=10)

# Bottom: deviation from arithmetic mean
ax = axes[1]
ref = arith_mean
for (label, curve), color in zip(list(consensus.items())[1:], colors_c[1:]):
    ax.plot(x, curve - ref, linewidth=1.2, color=color, label=f'{label} − Arithmetic Mean')
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Δ Intensity (a.u.)')
ax.set_title('Deviation from Arithmetic Mean')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### 6.6 Fingerprint Region Zoom (400–1800 cm⁻¹)

In [ ]:
mask_fp = (x >= 400) & (x <= 1800)

fig, ax = plt.subplots(figsize=(14, 5))
for (label, curve), color in zip(consensus.items(), colors_c):
    ax.plot(x[mask_fp], curve[mask_fp], linewidth=1.8, color=color, label=label)
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Consensus Spectra — Fingerprint Region (400–1800 cm⁻¹)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Save All Consensus Spectra

In [ ]:
consensus_df = pd.DataFrame(
    {label: curve for label, curve in consensus.items()},
    index=raman_shift
)
consensus_df.index.name = 'Raman Shift'

consensus_path = 'raman_consensus_spectra.xlsx'
consensus_df.to_excel(consensus_path)
print(f'Saved consensus spectra → {consensus_path}')
consensus_df.round(8).head()

## 8. Summary Statistics

In [ ]:
summary_rows = []
for label, curve in consensus.items():
    # Deviation from each individual spectrum
    diffs = np.linalg.norm(F - curve[:, None], axis=0)  # L2 distance to each spectrum
    summary_rows.append({
        'Method': label,
        'Peak (cm⁻¹)': round(float(x[np.argmax(curve)]), 1),
        'Peak Intensity': round(float(curve.max()), 6),
        'Mean L2 dist to spectra': round(float(diffs.mean()), 6),
        'Max L2 dist to spectra':  round(float(diffs.max()), 6),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Method')
print(summary_df.to_string())